# YOLO model local training

This notebook shows how to train a YOLO model locally. It also shows all the hyperparameters that can be tuned to control aspects related to the complexity and learning rate of the model, as well as stoping criteria.

In [ ]:
%pip install opencv-python matplotlib roboflow ultralytics

In [ ]:
from IPython import display
display.clear_output()

import cv2
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from roboflow import Roboflow

import ultralytics
from ultralytics import YOLO

ultralytics.checks()

In [ ]:
import torch
print(torch.backends.mps.is_available())

## Add tensorboard support
In your terminal run tensorboard --logdir runs/train and then open http://localhost:6006 in your browser. You'll see:

- Scalars: loss curves (box, cls, dfl), mAP, precision, recall per epoch
- Images: sample validation predictions with bounding boxes
- Graphs: the model architecture (if enabled)

Also, in your console, check that tensorboard is set to true, by running `yolo settings`. If it's set to false, run `yolo settings tensorboard=True`

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from ultralytics.utils.callbacks.tensorboard import callbacks as tb_callbacks

# Launch TensorBoard (run this in your terminal, pointing to your project folder):
# tensorboard --logdir runs/detect/runs/train/foe-bot-exp1

# TensorBoard is enabled automatically by Ultralytics when tensorboard is installed.
# Just make sure it's installed:
# pip install tensorboard

## Download dataset

In [ ]:
# Download the dataset from Roboflow, after labeling the images and creating the dataset.
# On Mac, permission issues may arise — fix them with:
# - sudo mkdir /Users/<your-username>/.config/roboflow   (create the config folder)
# - sudo chown -R <your-username>:staff ~/.config/roboflow  (grant your user ownership)
#roboflow.login()



rf = Roboflow(api_key="your_api_key")
project = rf.workspace("your_workspace").project("your_project_name")
version = project.version(2)
dataset = version.download("yolov8")

## Configuring the hyperparameters and training the model
Most impactful parameters to tune first: epochs, batch, imgsz, lr0, and freeze (if fine-tuning). The augmentation defaults are generally solid and rarely need changing unless your dataset is very small or domain-specific.

Apple Silicon note: device="mps" works well for YOLOv8 but amp=True can occasionally cause instability — if you see NaN losses, set amp=False.

Multi-GPU (Windows/Linux): Replace device='mps' with device=[0, 1] and consider increasing batch proportionally (e.g. 2 GPUs → double the batch size).

Early stopping tip: Setting patience=20–50 is a good practice to avoid wasting compute once the model has converged.

### Resume   
You can stop the training whenever you need because this process could take a long time. Underneath the training cell you will find a code to check if the weights exist after pasuing the training. And the next cell is to resume training. 



In [ ]:
# Load the model
# Full list of available pre-trained models: https://docs.ultralytics.com/models/yolov8/#performance-metrics
model = YOLO("yolov8m.pt")  # Load the pre-trained model weights

# ─────────────────────────────────────────────────────────────────────────────
# model.train() — Full hyperparameter reference
# ─────────────────────────────────────────────────────────────────────────────
results = model.train(

    # ── Dataset ──────────────────────────────────────────────────────────────
    data='Bottle-detection-2/data.yaml',  # Path to the dataset config file (YAML).
                                  # Must contain train/val/test paths and class names.

    # ── Core training settings ────────────────────────────────────────────────
    epochs=100,         # Total number of training epochs.
                        # More epochs = more learning, but risk of overfitting.
                        # Typical range: 50–300.

    patience=50,        # Early stopping: halt training if no improvement is seen
                        # for this many epochs. Set to 0 to disable.

    batch=16,           # Number of images per training batch.
                        # Higher = faster but needs more VRAM. Use -1 for AutoBatch.

    imgsz=640,          # Input image size (pixels). Images are resized to this square.
                        # Common values: 416, 512, 640, 1280.

    # ── Hardware ──────────────────────────────────────────────────────────────
    device="mps",       # Device to train on.
                        # "cpu"       → CPU only (slow)
                        # 0           → first CUDA GPU
                        # [0, 1]      → multi-GPU (CUDA)
                        # "mps"       → Apple Silicon GPU

    workers=8,          # Number of DataLoader worker threads for loading images.
                        # Reduce if you see memory errors or CPU bottlenecks.

    # ── Checkpointing & output ────────────────────────────────────────────────
    project="runs/train",   # Root folder where training results are saved.
    name="foe-bot-exp1",    # Sub-folder name for this specific run.
                            # Results go to <project>/<name>/.

    exist_ok=False,     # If False, a new numbered folder is created when the name
                        # already exists. If True, the existing folder is overwritten.

    save=True,          # Save checkpoints (best.pt and last.pt) during training.
    save_period=-1,     # Save a checkpoint every N epochs. -1 = disabled.
                        # Example: save_period=10 saves every 10 epochs.

    # ── Transfer learning ─────────────────────────────────────────────────────
    pretrained=True,    # Start from pre-trained weights (recommended).
                        # Set to False to train from scratch.

    freeze=0,           # Freeze the first N layers of the backbone (do not update
                        # their weights). Useful for fine-tuning.
                        # Example: freeze=10 freezes the first 10 layers.

    # ── Optimiser ─────────────────────────────────────────────────────────────
    optimizer="auto",   # Optimiser algorithm. Options:
                        # "SGD", "Adam", "AdamW", "NAdam", "RAdam", "RMSProp", "auto"
                        # "auto" selects SGD or AdamW based on the model.

    lr0=0.01,           # Initial learning rate.
                        # Lower values (e.g. 0.001) are safer for fine-tuning.

    lrf=0.01,           # Final learning rate as a fraction of lr0.
                        # The scheduler decays lr0 → lr0 * lrf over training.

    momentum=0.937,     # SGD momentum / Adam beta1.
                        # Controls how much past gradients influence the update.

    weight_decay=0.0005, # L2 regularisation penalty — discourages large weights
                         # and helps prevent overfitting.

    warmup_epochs=3.0,  # Number of epochs for learning-rate warm-up at the start.
                        # LR gradually rises from 0 to lr0 during this phase.

    warmup_momentum=0.8, # Initial momentum during the warm-up phase.
    warmup_bias_lr=0.1,  # Initial learning rate for bias parameters during warm-up.

    # ── Loss weights ──────────────────────────────────────────────────────────
    box=7.5,            # Weight for the bounding-box regression loss.
                        # Increase to make the model focus more on box accuracy.

    cls=0.5,            # Weight for the classification loss.
    dfl=1.5,            # Weight for the Distribution Focal Loss (box refinement).

    # ── Augmentation ──────────────────────────────────────────────────────────
    hsv_h=0.015,        # Random hue shift (fraction of 360°). Adds colour variety.
    hsv_s=0.7,          # Random saturation shift. Range: 0.0–1.0.
    hsv_v=0.4,          # Random brightness (value) shift. Range: 0.0–1.0.

    degrees=0.0,        # Random rotation range in degrees (e.g. 10 → ±10°).
    translate=0.1,      # Random translation as a fraction of image size (e.g. 0.1 = 10%).
    scale=0.5,          # Random scale factor (e.g. 0.5 means zoom between 50%–150%).
    shear=0.0,          # Random shear angle in degrees.
    perspective=0.0,    # Random perspective distortion. Range: 0.0–0.001.
    flipud=0.0,         # Probability of vertical flip. 0.0 = disabled.
    fliplr=0.5,         # Probability of horizontal flip. 0.5 = 50% chance per image.

    mosaic=1.0,         # Probability of Mosaic augmentation (combines 4 images).
                        # Very effective for small object detection. 0.0 = disabled.

    mixup=0.0,          # Probability of MixUp augmentation (blends 2 images).
                        # Useful for improving generalisation.

    copy_paste=0.0,     # Probability of Copy-Paste augmentation (pastes objects
                        # from one image onto another). Good for instance segmentation.

    # ── Evaluation ────────────────────────────────────────────────────────────
    val=True,           # Run validation after each epoch to track mAP, loss, etc.

    plots=True,         # Generate and save training plots (loss curves, confusion
                        # matrix, PR curve) inside the results folder.

    # ── Reproducibility ───────────────────────────────────────────────────────
    seed=0,             # Random seed for reproducibility. Use the same seed to get
                        # identical results across runs.

    deterministic=True, # Force deterministic CUDA operations.
                        # Ensures reproducibility but may slow down training slightly.

    # ── Misc ──────────────────────────────────────────────────────────────────
    resume=False,       # Resume training from the last saved checkpoint (last.pt).
                        # Set to the checkpoint path string to resume a specific run.

    amp=False,           # Automatic Mixed Precision (FP16). Speeds up training and
                        # reduces VRAM usage on supported hardware.

    fraction=1.0,       # Fraction of the training dataset to use.
                        # Example: 0.1 uses only 10% of images (useful for quick tests).

    profile=False,      # Profile ONNX and TensorRT speeds during training.
                        # Useful for benchmarking deployment targets.

    verbose=True,       # Print detailed training logs to the console.
)

In [ ]:
import os

weights_path = r"path_to_weights"

print(os.listdir(weights_path))

In [ ]:
from ultralytics import YOLO

model = YOLO(r"your_path_to_weights/last.pt")

results = model.train(resume=True)

# Inference

Now we'll use the trained model to predict on images on a folder. This can eventually be changed to aquire images from a camera in real time, or even for video. You will need to develop an app to showcase your usecase, and it can be based on this code. 

In [ ]:
from pathlib import Path
import cv2

INPUT_DIR = Path(r"your_path_to_images")
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

image_paths = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in SUPPORTED_EXTENSIONS
]

print("Found images:", len(image_paths))

for p in image_paths:
    img = cv2.imread(str(p))
    if img is None:
        print("BAD IMAGE:", p)
    else:
        print("OK:", p.name, img.shape)

In [ ]:
import json
from pathlib import Path

import cv2
from ultralytics import YOLO

MODEL_PATH = r"your_path_to_weights\best.pt"
INPUT_DIR = Path(r"your_path_to_images")
OUTPUT_DIR = Path(r"your_path_to_output_folder")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

model = YOLO(MODEL_PATH)

image_paths = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in SUPPORTED_EXTENSIONS
]

if not image_paths:
    print(f"No images found in '{INPUT_DIR}'. Supported formats: {SUPPORTED_EXTENSIONS}")

for image_path in image_paths:
    print(f"Processing: {image_path.name}")

    results = model.predict(
        source=str(image_path),
        conf=0.25,
        iou=0.7,
        imgsz=640,
        max_det=300,
        device="cpu",
        verbose=False,
    )

    result = results[0]

    detections = []

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections.append({
            "class_id": int(box.cls),
            "class_name": model.names[int(box.cls)],
            "confidence": round(float(box.conf), 4),
            "bbox": {
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "width": round(x2 - x1, 2),
                "height": round(y2 - y1, 2),
            }
        })

    json_path = OUTPUT_DIR / f"{image_path.stem}.json"

    with open(json_path, "w") as f:
        json.dump(detections, f, indent=2)

    annotated_image = result.plot(
        conf=True,
        labels=True,
        boxes=True,
        line_width=2,
    )

    output_image_path = OUTPUT_DIR / image_path.name
    cv2.imwrite(str(output_image_path), annotated_image)

    print(
        f"  → {len(detections)} detection(s) | "
        f"image saved to '{output_image_path}' | "
        f"JSON saved to '{json_path}'"
    )

print("\nDone.")